### Tools

In [60]:
### Load environment variables from .env file and set API keys for OpenAI, Groq, and Google
import os
## import load_dotenv from dotenv to load environment variables from .env file
from dotenv import load_dotenv
## import tool from langchain_core.tools to define tools
from langchain_core.tools import tool
## load environment variables from .env file
load_dotenv()
## Set API keys for OpenAI, Groq, and Google from environment variables
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### import init_chat_model from langchain to initialize chat models
from langchain.chat_models import init_chat_model
### import create_agent from langchain.agents to create agents
from langchain.agents import create_agent

##### Tool Binding Method

In [61]:
### Initialize the chat model using Groq
model = init_chat_model(
    model="qwen/qwen3-32b",
    model_provider="groq"
)

In [62]:
### Define Tool example weather with tool decorator
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

In [63]:
## bind the tool to the model
model_with_tools = model.bind_tools([get_weather])


In [64]:
## Get Response from tool
response = model_with_tools.invoke("What is the current weather in New York?")
print(response)

## Get Response from tool with a different location
for tool_call in response.tool_calls:
    #View the tool call details 
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified New York, I need to call that function with "New York" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'bxpmj7hx0', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 157, 'total_tokens': 255, 'completion_time': 0.136804829, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.006386736, 'prompt_tokens_details': None, 'queue_time': 0.049128033, 'total_time': 0.143191565}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_

##### Tool Defined in agent

In [65]:
### Initialize the chat model using Groq and Tools

agent = create_agent(
    model=model,
    tools=[get_weather]
)

In [66]:
## Get Response from tool
response = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in London?"}]})
response["messages"][-1].content

'The current weather in London is sunny with a temperature of 25°C.'

##### Tool Execution Loop

In [67]:
# Step 1 - Model Generates Tool Calls
messages = [{"role": "user", "content": "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2 - Execute Tools and Collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool call with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3 - Pass results back to the model for final response generation
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in Boston is **sunny** with a temperature of **25°C**. ☀️ Let me know if you need more details!


In [70]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'fmkyw4h6s', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 155, 'total_tokens': 245, 'completion_time': 0.131175879, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.007511397, 'prompt_tokens_details': None, 'queue_time': 0.049363612, 'total_time': 0.138687276}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_rea